In [1]:
import obd
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import pygame
import math

def draw_dashboard(screen, rpm, speed):
    """
    Renders a needle gauge for RPM and text for Speed.
    rpm: float/int, speed: float/int
    """
    # Configuration
    CENTER = (400, 300)
    RADIUS = 200
    MAX_RPM = 8000
    BG_COLOR = (20, 20, 20)
    NEEDLE_COLOR = (255, 50, 50)

    # 1. Clear the area (or the whole screen)
    screen.fill(BG_COLOR) 

    # 2. Calculate Needle Angle
    # Map RPM (0-8000) to Degrees (135-405)
    angle = 135 + (min(rpm, MAX_RPM) / MAX_RPM) * 270
    radians = math.radians(angle)

    # 3. Calculate Needle End Point
    end_x = CENTER[0] + RADIUS * math.cos(radians)
    end_y = CENTER[1] + RADIUS * math.sin(radians)

    # 4. Draw Gauge Elements
    # Draw Background Arc
    rect = [CENTER[0]-RADIUS, CENTER[1]-RADIUS, RADIUS*2, RADIUS*2]
    pygame.draw.arc(screen, (150, 150, 150), rect, math.radians(135), math.radians(405), 4)

    # Draw the Needle
    pygame.draw.line(screen, NEEDLE_COLOR, CENTER, (end_x, end_y), 6)
    
    # Draw Speed Text in the center
    font = pygame.font.SysFont("Arial", 50, bold=True)
    speed_surf = font.render(f"{int(speed)} km/h", True, (255, 255, 255))
    screen.blit(speed_surf, (CENTER[0] - speed_surf.get_width()//2, CENTER[1] + 50))

In [3]:
import pygame
import obd
import math
import sys

# --- 2. LIVE DATA VARIABLES ---
current_rpm = 0
current_speed = 0

# These "callback" functions update our variables whenever the car has new data
def update_rpm(r):
    global current_rpm
    if not r.is_null():
        current_rpm = r.value.magnitude

def update_speed(s):
    global current_speed
    if not s.is_null():
        current_speed = s.value.magnitude

# --- 3. OBD SETUP ---
# connection = obd.Async() # Auto-connect
connection = obd.Async(fast=True) 

connection.watch(obd.commands.RPM, callback=update_rpm)
connection.watch(obd.commands.SPEED, callback=update_speed)
connection.start()

# --- 4. PYGAME INITIALIZATION ---
pygame.init()
screen = pygame.display.set_mode((800, 600))
clock = pygame.time.Clock()

# --- 5. THE MAIN LIVE LOOP ---
running = True
try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        # Use the same function, passing the latest live variables
        draw_dashboard(screen, current_rpm, current_speed)
        
        pygame.display.flip()
        
        # We run the UI at 30-60fps for smoothness, 
        # even if OBD only updates 5-10 times per second.
        clock.tick(60) 

finally:
    # Always stop the OBD thread and close Pygame properly
    connection.stop()
    pygame.quit()
    sys.exit()

SystemExit: 

c:\Users\jaros\miniconda3\envs\obd\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
connection.stop()